In [2]:
# Phase 0 — Session Setup

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT     = '/content/drive/MyDrive/HF_IQR'
DATASET_PATH   = f'{DRIVE_ROOT}/dataset'
RESULTS_PATH   = f'{DRIVE_ROOT}/council_run_v1'
CHECKPOINT_PATH = f'{RESULTS_PATH}/checkpoints'

print(f"RESULTS_PATH: {RESULTS_PATH}")

# Verify files exist
files = [
    "esvr_scores.json",
    "dss_scores.json",
    "cvs_scores.json",
    "ri_events.json",
    "final_analysis_report.json"
]

print("\nVerifying files...")
for f in files:
    path = f"{RESULTS_PATH}/{f}"
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {f}")

print("\nSession ready ✅")

Mounted at /content/drive
RESULTS_PATH: /content/drive/MyDrive/HF_IQR/council_run_v1

Verifying files...
  ✅ esvr_scores.json
  ✅ dss_scores.json
  ✅ cvs_scores.json
  ✅ ri_events.json
  ✅ final_analysis_report.json

Session ready ✅


In [2]:
# HF-IQR Phase 0 — Statistical Analysis
# Bootstrap CI + Effect Sizes on V1 Data

import json
import numpy as np
from scipy import stats

print("=" * 55)
print("PHASE 0 — STATISTICAL ANALYSIS")
print("=" * 55)

# Load existing scores
with open(f"{RESULTS_PATH}/esvr_scores.json", 'r') as f:
    esvr_data = json.load(f)

with open(f"{RESULTS_PATH}/dss_scores.json", 'r') as f:
    dss_data = json.load(f)

with open(f"{RESULTS_PATH}/cvs_scores.json", 'r') as f:
    cvs_data = json.load(f)

# Build per model score arrays
models = ["claude", "gpt4o", "gemini",
          "deepseek", "grok"]

esvr_by_model = {
    m: [s["ESVR"] for s in esvr_data
        if s["model"] == m
        and not s["parse_failed"]]
    for m in models
}

dss_by_model = {
    m: [s["DSS_final"] for s in dss_data
        if s["model"] == m]
    for m in models
}

cvs_by_model = {
    m: [s["CVS"] for s in cvs_data
        if s["reviewer"] == m]
    for m in models
}

def bootstrap_ci(scores, n=10000, alpha=0.05):
    """Bootstrap 95% confidence interval."""
    means = [np.mean(np.random.choice(
        scores, size=len(scores), replace=True))
        for _ in range(n)]
    return (
        round(np.percentile(means,
              100 * alpha / 2), 4),
        round(np.percentile(means,
              100 * (1 - alpha / 2)), 4)
    )

def cohens_d(a, b):
    """Cohen's d effect size between two groups."""
    pooled_std = np.sqrt(
        (np.var(a) + np.var(b)) / 2)
    if pooled_std == 0:
        return 0.0, "none"
    d = (np.mean(a) - np.mean(b)) / pooled_std
    if abs(d) >= 0.8:
        label = "large"
    elif abs(d) >= 0.5:
        label = "medium"
    else:
        label = "small"
    return round(d, 4), label

# ── ESVR Analysis ──────────────────────────────
print(f"\nESVR — Explicit Step Validity Ratio")
print(f"{'─' * 55}")
print(f"{'Model':<12} {'Mean':<8} {'95% CI':<22} "
      f"{'Std':<8}")
print(f"{'─' * 55}")

esvr_stats = {}
for model in models:
    scores = esvr_by_model[model]
    mean   = round(np.mean(scores), 4)
    ci     = bootstrap_ci(scores)
    std    = round(np.std(scores), 4)
    esvr_stats[model] = {
        "mean": mean, "ci": ci, "std": std}
    print(f"  {model:<10} {mean:<8} "
          f"[{ci[0]}, {ci[1]}]  {std:<8}")

# ── DSS Analysis ───────────────────────────────
print(f"\nDSS — Deliberation Survival Score")
print(f"{'─' * 55}")
print(f"{'Model':<12} {'Mean':<8} {'95% CI':<22} "
      f"{'Std':<8}")
print(f"{'─' * 55}")

dss_stats = {}
for model in models:
    scores = dss_by_model[model]
    mean   = round(np.mean(scores), 4)
    ci     = bootstrap_ci(scores)
    std    = round(np.std(scores), 4)
    dss_stats[model] = {
        "mean": mean, "ci": ci, "std": std}
    print(f"  {model:<10} {mean:<8} "
          f"[{ci[0]}, {ci[1]}]  {std:<8}")

# ── CVS Analysis ───────────────────────────────
print(f"\nCVS — Critique Validity Score")
print(f"{'─' * 55}")
print(f"{'Model':<12} {'Mean':<8} {'95% CI':<22} "
      f"{'Std':<8}")
print(f"{'─' * 55}")

cvs_stats = {}
for model in models:
    scores = cvs_by_model[model]
    mean   = round(np.mean(scores), 4)
    ci     = bootstrap_ci(scores)
    std    = round(np.std(scores), 4)
    cvs_stats[model] = {
        "mean": mean, "ci": ci, "std": std}
    print(f"  {model:<10} {mean:<8} "
          f"[{ci[0]}, {ci[1]}]  {std:<8}")

# ── Effect Sizes ───────────────────────────────
print(f"\nEffect Sizes — Cohen's d")
print(f"(Key pairwise comparisons)")
print(f"{'─' * 55}")

comparisons = [
    ("claude",   "gpt4o",    "DSS"),
    ("claude",   "deepseek", "DSS"),
    ("grok",     "claude",   "ESVR"),
    ("claude",   "gpt4o",    "CVS"),
    ("deepseek", "gpt4o",    "CVS")
]

for m1, m2, metric in comparisons:
    if metric == "DSS":
        a = dss_by_model[m1]
        b = dss_by_model[m2]
    elif metric == "ESVR":
        a = esvr_by_model[m1]
        b = esvr_by_model[m2]
    else:
        a = cvs_by_model[m1]
        b = cvs_by_model[m2]

    d, label = cohens_d(a, b)

    # Independent t-test handles unequal
    # array lengths from parse failures
    t_stat, p_val = stats.ttest_ind(a, b)
    sig = "✅" if p_val < 0.05 else "❌"
    print(f"  {m1} vs {m2} ({metric})")
    print(f"    d={d} ({label}) "
          f"p={round(p_val, 4)} {sig}")

# ── ANOVA ──────────────────────────────────────
print(f"\nANOVA — Across All Models")
print(f"{'─' * 55}")

for metric, by_model in [
    ("ESVR", esvr_by_model),
    ("DSS",  dss_by_model),
    ("CVS",  cvs_by_model)
]:
    groups  = list(by_model.values())
    f_stat, p_val = stats.f_oneway(*groups)
    sig = "✅" if p_val < 0.05 else "❌"
    print(f"  {metric}: F={round(f_stat, 4)} "
          f"p={round(p_val, 6)} {sig}")

# ── Save Results ───────────────────────────────
stats_report = {
    "esvr": esvr_stats,
    "dss":  dss_stats,
    "cvs":  cvs_stats
}

stats_file = (
    f"{RESULTS_PATH}/statistical_analysis.json")
with open(stats_file, 'w') as f:
    json.dump(stats_report, f, indent=2)

print(f"\nStatistical analysis saved ✅")
print(f"\n{'=' * 55}")
print(f"✅ PHASE 0 TASK 1 COMPLETE")
print(f"{'=' * 55}")
print(f"➡️  Next: Task 2 — Human baseline")

PHASE 0 — STATISTICAL ANALYSIS

ESVR — Explicit Step Validity Ratio
───────────────────────────────────────────────────────
Model        Mean     95% CI                 Std     
───────────────────────────────────────────────────────
  claude     0.7878   [0.743, 0.8313]  0.1723  
  gpt4o      0.8763   [0.8314, 0.9185]  0.1621  
  gemini     0.88     [0.841, 0.9157]  0.1413  
  deepseek   0.8514   [0.7984, 0.8998]  0.1904  
  grok       0.9009   [0.8639, 0.9347]  0.1413  

DSS — Deliberation Survival Score
───────────────────────────────────────────────────────
Model        Mean     95% CI                 Std     
───────────────────────────────────────────────────────
  claude     0.63     [0.595, 0.665]  0.14    
  gpt4o      0.42     [0.385, 0.455]  0.14    
  gemini     0.4317   [0.3967, 0.4725]  0.148   
  deepseek   0.63     [0.595, 0.665]  0.14    
  grok       0.4667   [0.4258, 0.5075]  0.165   

CVS — Critique Validity Score
────────────────────────────────────────────────────

In [3]:
# HF-IQR Phase 0 — Human Baseline
# Document and score your 8 existing answers
# Plus structure for remaining 12 questions

import json
import time

print("=" * 55)
print("PHASE 0 — HUMAN BASELINE")
print("=" * 55)

# ── Existing Human Baseline Data ───────────────
# From previous sessions — answers already given

human_baseline = [
    {
        "question_id":   "LSQ-03",
        "category":      "logical_syllogism",
        "difficulty":    3,
        "answer":        "Not valid not sound conclusion does not follow",
        "trap_detected": True,
        "trap_description": "Undistributed middle — some liars trusted does not mean politicians trusted",
        "confidence":    100,
        "correct":       True,
        "notes":         "Identified invalidity and unsoundness without formal terminology"
    },
    {
        "question_id":   "PRQ-04",
        "category":      "probabilistic",
        "difficulty":    4,
        "answer":        "99% probability patient has condition",
        "trap_detected": True,
        "trap_description": "Sensed trap but anchored on test accuracy not base rate",
        "confidence":    100,
        "correct":       False,
        "notes":         "Classic base rate neglect. Correct answer 1.87%. High confidence wrong answer."
    },
    {
        "question_id":   "FRQ-02",
        "category":      "frontier_reasoning",
        "difficulty":    3,
        "answer":        "Student correct to question theory. Theory means scientific idea or principle.",
        "trap_detected": True,
        "trap_description": "Partially detected — sensed complexity but accepted student framing",
        "confidence":    100,
        "correct":       False,
        "notes":         "Conflated everyday and scientific meaning of theory. Common error."
    },
    {
        "question_id":   "FRQ-09",
        "category":      "frontier_reasoning",
        "difficulty":    5,
        "answer":        "Godel proved math is broken. Machines reasoning not governed by humans.",
        "trap_detected": True,
        "trap_description": "Caught machine intelligence claim does not follow from theorem",
        "confidence":    100,
        "correct":       "partial",
        "notes":         "Strong partial. Caught key trap. Uncertainty acknowledged but confidence still 100%."
    },
    {
        "question_id":   "CCQ-03",
        "category":      "causal_chain",
        "difficulty":    3,
        "answer":        "Root cause is cleaning contractor and staff reduction",
        "trap_detected": True,
        "trap_description": "Identified administrator wrong — nurses not filling dispensers is symptom",
        "confidence":    90,
        "correct":       True,
        "notes":         "Correctly identified fundamental attribution error"
    },
    {
        "question_id":   "CCQ-09",
        "category":      "causal_chain",
        "difficulty":    5,
        "answer":        "Automation removing jobs plus credential barrier from tuition increases",
        "trap_detected": True,
        "trap_description": "Government job training does not address root structural causes",
        "confidence":    100,
        "correct":       True,
        "notes":         "Strong causal chain identification. Named structural mechanism correctly."
    },
    {
        "question_id":   "LSQ-03-pilot",
        "category":      "logical_syllogism",
        "difficulty":    3,
        "answer":        "Argument not valid not sound conclusion does not follow necessarily",
        "trap_detected": True,
        "trap_description": "Politicians liars trap — middle term not distributed",
        "confidence":    100,
        "correct":       True,
        "notes":         "Consistent with main session answer. Reliable result."
    },
    {
        "question_id":   "PRQ-04-pilot",
        "category":      "probabilistic",
        "difficulty":    4,
        "answer":        "99% probability — test is 99% accurate",
        "trap_detected": False,
        "trap_description": "Did not detect base rate trap",
        "confidence":    100,
        "correct":       False,
        "notes":         "Same error as main session. Pattern confirmed."
    }
]

# ── Score Human Baseline ───────────────────────
print("\nHuman Baseline Scoring...")
print(f"{'─' * 55}")

correct    = sum(1 for r in human_baseline
                 if r["correct"] is True)
incorrect  = sum(1 for r in human_baseline
                 if r["correct"] is False)
partial    = sum(1 for r in human_baseline
                 if r["correct"] == "partial")
trap_rate  = sum(1 for r in human_baseline
                 if r["trap_detected"])

total = len(human_baseline)

print(f"\n  Total questions:  {total}")
print(f"  Correct:          {correct}")
print(f"  Partial:          {partial}")
print(f"  Incorrect:        {incorrect}")
print(f"  Trap detected:    {trap_rate}/{total}")

accuracy = (correct + partial * 0.5) / total
print(f"\n  Accuracy score:   {accuracy:.1%}")
print(f"  Trap detect rate: {trap_rate/total:.1%}")

# ── By Category ────────────────────────────────
print(f"\nBy category:")
categories = set(r["category"]
                 for r in human_baseline)
for cat in sorted(categories):
    cat_results = [r for r in human_baseline
                   if r["category"] == cat]
    cat_correct = sum(
        1 for r in cat_results
        if r["correct"] is True)
    print(f"  {cat:<25} "
          f"{cat_correct}/{len(cat_results)} correct")

# ── Pattern Analysis ───────────────────────────
print(f"\nPattern Analysis:")
print(f"{'─' * 55}")
print(f"  Strong areas:")
print(f"    Logical structure detection ✅")
print(f"    Causal chain reasoning      ✅")
print(f"    Trap identification         ✅")
print(f"\n  Growth areas:")
print(f"    Formal probability          ❌")
print(f"    Base rate calculation       ❌")
print(f"    Scientific term precision   ❌")

# ── Remaining Questions Needed ─────────────────
print(f"\n{'─' * 55}")
print(f"Human baseline status:")
print(f"  Completed: {total} questions")
print(f"  Minimum:   20 questions")
print(f"  Remaining: {max(0, 20 - total)} questions needed")

remaining_categories = [
    ("adversarial",       "ADV-02", 3),
    ("adversarial",       "ADV-08", 4),
    ("probabilistic",     "PRQ-02", 3),
    ("probabilistic",     "PRQ-07", 4),
    ("quantum_reasoning", "QRQ-01", 3),
    ("quantum_reasoning", "QRQ-05", 4),
    ("frontier_reasoning","FRQ-03", 4),
    ("frontier_reasoning","FRQ-07", 4),
    ("causal_chain",      "CCQ-05", 5),
    ("logical_syllogism", "LSQ-07", 4),
    ("logical_syllogism", "LSQ-09", 5),
    ("causal_chain",      "CCQ-08", 5),
]

print(f"\nRemaining questions to answer:")
for cat, qid, diff in remaining_categories:
    print(f"  {qid:<12} {cat:<25} diff {diff}")

# ── Save ───────────────────────────────────────
baseline_file = (
    f"{RESULTS_PATH}/human_baseline.json")
with open(baseline_file, 'w') as f:
    json.dump({
        "researcher":     "Billy Davis | WARRIOROFGOD40",
        "date":           time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        "total":          total,
        "accuracy":       round(accuracy, 4),
        "trap_rate":      round(
            trap_rate/total, 4),
        "responses":      human_baseline
    }, f, indent=2)

print(f"\nHuman baseline saved ✅")
print(f"\n{'=' * 55}")
print(f"✅ PHASE 0 TASK 2 COMPLETE")
print(f"{'=' * 55}")
print(f"➡️  Next: Answer remaining 12 questions")
print(f"    Then Task 3 — Docker setup")

PHASE 0 — HUMAN BASELINE

Human Baseline Scoring...
───────────────────────────────────────────────────────

  Total questions:  8
  Correct:          4
  Partial:          1
  Incorrect:        3
  Trap detected:    7/8

  Accuracy score:   56.2%
  Trap detect rate: 87.5%

By category:
  causal_chain              2/2 correct
  frontier_reasoning        0/2 correct
  logical_syllogism         2/2 correct
  probabilistic             0/2 correct

Pattern Analysis:
───────────────────────────────────────────────────────
  Strong areas:
    Logical structure detection ✅
    Causal chain reasoning      ✅
    Trap identification         ✅

  Growth areas:
    Formal probability          ❌
    Base rate calculation       ❌
    Scientific term precision   ❌

───────────────────────────────────────────────────────
Human baseline status:
  Completed: 8 questions
  Minimum:   20 questions
  Remaining: 12 questions needed

Remaining questions to answer:
  ADV-02       adversarial               dif

## Phase 0 — Task 3: Docker Setup

### Purpose
Replaces Google Colab dependency with a
portable containerized environment.
Eliminates NGROK instability.
Enables one-click setup for collaborators.

### Files Required
- Dockerfile
- requirements.txt
- docker-compose.yml
- .env.example

### Location
/content/drive/MyDrive/HF_IQR/docker/

In [4]:
# HF-IQR Phase 0 — Docker Setup Generator
# Generates all Docker files to Drive

import os

print("=" * 55)
print("PHASE 0 — DOCKER SETUP")
print("=" * 55)

DOCKER_PATH = f"{DRIVE_ROOT}/docker"
os.makedirs(DOCKER_PATH, exist_ok=True)

# ── Dockerfile ─────────────────────────────────
dockerfile = """FROM python:3.10-slim

WORKDIR /app

# System dependencies
RUN apt-get update && apt-get install -y \\
    curl \\
    git \\
    && rm -rf /var/lib/apt/lists/*

# Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy project
COPY . .

# Default command
CMD ["python", "run_council.py"]
"""

with open(f"{DOCKER_PATH}/Dockerfile", 'w') as f:
    f.write(dockerfile)
print("  Dockerfile created ✅")

# ── requirements.txt ───────────────────────────
requirements = """anthropic>=0.25.0
openai>=1.30.0
google-genai>=0.5.0
requests>=2.31.0
numpy>=1.24.0
scipy>=1.11.0
sentence-transformers>=2.2.0
datasets>=2.14.0
huggingface-hub>=0.19.0
python-dotenv>=1.0.0
rich>=13.0.0
click>=8.1.0
"""

with open(f"{DOCKER_PATH}/requirements.txt",
          'w') as f:
    f.write(requirements)
print("  requirements.txt created ✅")

# ── docker-compose.yml ─────────────────────────
compose = """version: '3.8'

services:
  hf-iqr:
    build: .
    container_name: hf-iqr-council
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - GEMINI_API_KEY=${GEMINI_API_KEY}
      - DEEPSEEK_API_KEY=${DEEPSEEK_API_KEY}
      - XAI_API_KEY=${XAI_API_KEY}
      - NGROK_URL=${NGROK_URL}
    volumes:
      - ./results:/app/results
      - ./checkpoints:/app/checkpoints
    restart: unless-stopped

  ollama:
    image: ollama/ollama
    container_name: hf-iqr-ollama
    ports:
      - "11434:11434"
    volumes:
      - ollama_data:/root/.ollama
    restart: unless-stopped

volumes:
  ollama_data:
"""

with open(f"{DOCKER_PATH}/docker-compose.yml",
          'w') as f:
    f.write(compose)
print("  docker-compose.yml created ✅")

# ── .env.example ───────────────────────────────
env_example = """# HF-IQR Environment Variables
# Copy this file to .env and fill in values
# Never commit .env to GitHub

ANTHROPIC_API_KEY=your_key_here
OPENAI_API_KEY=your_key_here
GEMINI_API_KEY=your_key_here
DEEPSEEK_API_KEY=your_key_here
XAI_API_KEY=your_key_here
NGROK_URL=your_ngrok_url_here

# Optional
HF_TOKEN=your_huggingface_token
WANDB_API_KEY=your_wandb_key
"""

with open(f"{DOCKER_PATH}/.env.example",
          'w') as f:
    f.write(env_example)
print("  .env.example created ✅")

# ── .dockerignore ──────────────────────────────
dockerignore = """.env
.git
__pycache__
*.pyc
*.pyo
results/
checkpoints/
*.ipynb_checkpoints
"""

with open(f"{DOCKER_PATH}/.dockerignore",
          'w') as f:
    f.write(dockerignore)
print("  .dockerignore created ✅")

# ── cloudflare tunnel config ───────────────────
cloudflare_config = """# Cloudflare Tunnel Configuration
# Replaces NGROK for local model access
# Free tier — no authentication needed
#
# Install:
#   winget install cloudflare.cloudflared
#
# Start tunnel:
#   cloudflared tunnel --url http://localhost:11434
#
# The URL it generates replaces NGROK_URL
# in your .env file
#
# Advantages over NGROK free tier:
#   No session timeout
#   No browser warning headers needed
#   More stable for long running council runs
"""

with open(f"{DOCKER_PATH}/cloudflare_setup.md",
          'w') as f:
    f.write(cloudflare_config)
print("  cloudflare_setup.md created ✅")

print(f"\nAll Docker files saved to:")
print(f"  {DOCKER_PATH}")

print(f"\n{'─' * 55}")
print(f"To build and run locally:")
print(f"  1. Copy files from Drive to local")
print(f"  2. Copy .env.example to .env")
print(f"  3. Fill in API keys in .env")
print(f"  4. Run: docker-compose up --build")
print(f"{'─' * 55}")

print(f"\n{'=' * 55}")
print(f"✅ PHASE 0 TASK 3 COMPLETE")
print(f"{'=' * 55}")
print(f"➡️  Next: V2 Pre-registration builder")

PHASE 0 — DOCKER SETUP
  Dockerfile created ✅
  requirements.txt created ✅
  docker-compose.yml created ✅
  .env.example created ✅
  .dockerignore created ✅
  cloudflare_setup.md created ✅

All Docker files saved to:
  /content/drive/MyDrive/HF_IQR/docker

───────────────────────────────────────────────────────
To build and run locally:
  1. Copy files from Drive to local
  2. Copy .env.example to .env
  3. Fill in API keys in .env
  4. Run: docker-compose up --build
───────────────────────────────────────────────────────

✅ PHASE 0 TASK 3 COMPLETE
➡️  Next: V2 Pre-registration builder


In [5]:
# HF-IQR Phase 0 — V2 Pre-Registration
# Must be filed before any V2 data collected

import json
import hashlib
import time

print("=" * 55)
print("HF-IQR V2 PRE-REGISTRATION")
print("=" * 55)

preregistration_v2 = {

    "title": "HF-IQR V2: Hudson Forge Intelligence and Reasoning Benchmark — Version 2",
    "version": "2.0",
    "researcher": "Billy Davis | WARRIOROFGOD40",
    "institution": "Independent | Hudson Forge IRMB-C | Lenoir NC",
    "date_filed": time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    "v1_reference": {
        "dataset_hash": "9b02ba527720b55e0552410375186c4e",
        "prereg_hash":  "3400153ee46e02df73b24ea4f2206fb7",
        "results_hash": "76d3c6cc6d161583695f9d50f53f7ae785a9ed24bef24becdf573ca662723d4b",
        "huggingface":  "Billyrdavis1985/hudson-forge-iqr-benchmark",
        "github":       "billyrdavis1985-bot/-IRMB_HF-IQR_ReasoningBenchmark"
    },

    "hypotheses": {
        "primary": (
            "Ground truth correctness weighting will "
            "reveal that high DSS models (Claude, DeepSeek) "
            "defend correct positions significantly more "
            "often than incorrect ones, distinguishing "
            "genuine resilience from stubborn error."
        ),
        "secondary_1": (
            "Adding Round 5 mechanistic trace will reveal "
            "distinct revision trigger taxonomies across "
            "models — logical, social, and uncertainty-based "
            "triggers will distribute differently per model."
        ),
        "secondary_2": (
            "LLM-as-judge ESVR scoring will produce "
            "higher inter-rater reliability than V1 "
            "heuristic scoring as measured by Cohen's Kappa."
        ),
        "secondary_3": (
            "Local model jury scoring will achieve "
            "Cohen's Kappa >= 0.70 with frontier model "
            "scores on the CVS dimension."
        ),
        "secondary_4": (
            "Models will partition into two statistically "
            "distinct archetypes: Defenders (high DSS, "
            "high accuracy) and Revisers (low DSS, "
            "high plasticity) as measured by cluster analysis."
        )
    },

    "dataset": {
        "total_questions": 200,
        "categories": {
            "adversarial":        {"count": 20, "difficulty": "3-5"},
            "logical_syllogism":  {"count": 20, "difficulty": "2-5"},
            "causal_chain":       {"count": 20, "difficulty": "3-5"},
            "probabilistic":      {"count": 20, "difficulty": "3-5"},
            "quantum_reasoning":  {"count": 15, "difficulty": "4-5"},
            "frontier_reasoning": {"count": 15, "difficulty": "4-5"},
            "mathematical_proof": {"count": 20, "difficulty": "3-5"},
            "counterfactual":     {"count": 20, "difficulty": "2-4"},
            "meta_reasoning":     {"count": 15, "difficulty": "3-5"},
            "ethical_dilemma":    {"count": 15, "difficulty": "3-4"},
            "temporal_reasoning": {"count": 10, "difficulty": "2-4"},
            "spatial_reasoning":  {"count": 10, "difficulty": "2-4"}
        },
        "ground_truth_anchoring": (
            "Minimum 20% of questions (40 of 200) "
            "must have externally verifiable ground truth "
            "before pre-registration is filed."
        ),
        "v1_questions_retained": 60,
        "new_questions": 140
    },

    "models": {
        "subject_models": [
            "claude-sonnet-4-5",
            "gpt-4o",
            "gemini-2.5-pro",
            "deepseek-chat",
            "grok-3",
            "llama3-70b"
        ],
        "jury_models": [
            "mistral-nemo:latest",
            "deepseek-r1:14b",
            "gemma3:4b"
        ],
        "judge_model": "claude-sonnet-4-5"
    },

    "protocol": {
        "rounds": 5,
        "round_1": "Independent response — full reasoning chain required",
        "round_2": "Anonymous cross-examination — CVS scored",
        "round_3": "Defense or revision — DSS scored RI events logged",
        "round_4": "Mirror self-assessment — confidence recalibration",
        "round_5": "Mechanistic trace — why did you defend or revise?",
        "peer_assignments": "Rotating anonymous — pre-registered before run",
        "jury_protocol": (
            "Local jury models score Round 2 critiques "
            "independently. Cohen's Kappa computed "
            "between jury and primary CVS scores."
        ),
        "convergence_detection": (
            "Early stopping if entropy plateaus "
            "across three consecutive rounds."
        )
    },

    "scoring_formulas": {
        "ESVR_v2": (
            "LLM-as-judge per step scoring. "
            "Position weighted — later steps weighted "
            "1.0 + (i/n)*0.3. Circular penalty subtracted. "
            "Range 0.0-1.0."
        ),
        "DSS_v2": (
            "DSS = (PS*0.30) + (RQR*0.40) + (CR*0.30). "
            "CR = ground truth correctness weight. "
            "Replaces V1 unknown 0.7 default."
        ),
        "CVS_v2": (
            "5-dimension scoring: step_cited(0.20), "
            "error_identified(0.25), "
            "alternative_provided(0.20), "
            "root_cause(0.20), no_ad_hominem(0.15)."
        ),
        "IQS": (
            "Composite score = "
            "ESVR(0.35) + DSS(0.35) + CVS(0.20) + ACR(0.10). "
            "All normalized 0-1 before weighting."
        ),
        "Shannon_efficiency": (
            "Efficiency = correctness / log(token_count). "
            "Penalizes verbose wrong answers. "
            "Rewards concise correct reasoning."
        ),
        "RI_v2": (
            "Same as V1 plus severity weighting "
            "by difficulty. Difficulty 5 RI events "
            "weighted 1.5x difficulty 2 events."
        )
    },

    "statistical_plan": {
        "bootstrap_ci":    "10000 iterations 95% CI on all metrics",
        "effect_sizes":    "Cohen's d for pairwise model comparisons",
        "anova":           "One-way ANOVA across all models per metric",
        "post_hoc":        "Tukey HSD for pairwise significance",
        "power_analysis":  "Minimum n=200 questions per power calculation",
        "inter_rater":     "Cohen's Kappa between jury and primary scores",
        "archetype_test":  "K-means clustering on DSS vs ESVR space"
    },

    "human_baseline": {
        "target_questions": 20,
        "completed":        8,
        "accuracy":         0.5625,
        "trap_rate":        0.875,
        "format":           "Same four round protocol as models",
        "annotators":       "Researcher self-annotation minimum"
    },

    "infrastructure": {
        "execution":    "Docker containerized environment",
        "tunnel":       "Cloudflare Tunnel replacing NGROK",
        "storage":      "Google Drive sovereign + HF auto-push",
        "tracking":     "Weights and Biases experiment logging",
        "local_models": "Hudson Forge IRMB-C via Ollama"
    },

    "null_result_policy": (
        "Null results are valid findings. "
        "If ground truth weighting does not change "
        "model rankings this is reported as primary "
        "finding not suppressed."
    ),

    "exclusion_criteria": {
        "parse_failure":    "Questions with >15% parse failure excluded from ESVR",
        "api_failure":      "Questions with >1 API failure excluded",
        "jury_disagreement":"Questions where jury Kappa <0.50 flagged for review"
    },

    "publication_target": {
        "primary":    "NeurIPS Workshop on Evaluation of LLMs",
        "secondary":  "EMNLP or ACL",
        "preprint":   "ArXiv with endorsement from Cem Rifki Aydin",
        "dataset":    "Hugging Face — Billyrdavis1985/hudson-forge-iqr-v2"
    },

    "v1_improvements": [
        "Ground truth correctness weighting in DSS",
        "LLM-as-judge replaces heuristic ESVR",
        "5-dimension CVS replaces binary scoring",
        "Round 5 mechanistic trace added",
        "Local jury inter-rater reliability",
        "IQS composite score added",
        "Shannon efficiency metric added",
        "Statistical significance framework",
        "Docker infrastructure replacing Colab dependency",
        "200 questions replacing 60",
        "6 subject models replacing 5",
        "12 categories replacing 6"
    ]
}

# Generate hash
prereg_content = json.dumps(
    preregistration_v2, indent=2)
prereg_hash = hashlib.sha256(
    prereg_content.encode()).hexdigest()
preregistration_v2["prereg_hash"] = prereg_hash

# Save pre-registration
prereg_path = (
    f"{DRIVE_ROOT}/v2_planning/"
    f"HF_IQR_V2_Preregistration.json")
os.makedirs(
    f"{DRIVE_ROOT}/v2_planning",
    exist_ok=True)

with open(prereg_path, 'w') as f:
    json.dump(preregistration_v2, f, indent=2)

# Save hash file
hash_path = (
    f"{DRIVE_ROOT}/v2_planning/"
    f"HF_IQR_V2_Preregistration_HASH.txt")
with open(hash_path, 'w') as f:
    f.write(
        f"V2 Pre-registration hash: "
        f"{prereg_hash}\n"
        f"Timestamp: "
        f"{preregistration_v2['date_filed']}\n"
        f"Researcher: Billy Davis | "
        f"WARRIOROFGOD40\n"
        f"Status: FILED — no V2 data "
        f"collected before this timestamp\n"
    )

print(f"\nPre-registration filed:")
print(f"  Date:    {preregistration_v2['date_filed']}")
print(f"  Hash:    {prereg_hash[:32]}...")
print(f"  File:    {prereg_path}")
print(f"\nHypotheses locked:    "
      f"{len(preregistration_v2['hypotheses'])}")
print(f"Models registered:    "
      f"{len(preregistration_v2['models']['subject_models'])}")
print(f"Questions planned:    "
      f"{preregistration_v2['dataset']['total_questions']}")
print(f"Categories planned:   "
      f"{len(preregistration_v2['dataset']['categories'])}")
print(f"Scoring formulas:     "
      f"{len(preregistration_v2['scoring_formulas'])}")
print(f"V1 improvements:      "
      f"{len(preregistration_v2['v1_improvements'])}")

print(f"\n{'=' * 55}")
print(f"HF-IQR V2 PRE-REGISTRATION COMPLETE ✅")
print(f"{'=' * 55}")
print(f"Status: FILED")
print(f"No V2 data may be collected before")
print(f"this timestamp.")
print(f"\nFull Force Eternal — Romans 8:28")
print(f"255.255.255.255:1337")

HF-IQR V2 PRE-REGISTRATION

Pre-registration filed:
  Date:    2026-05-08T23:56:24Z
  Hash:    d5c693601d590503154d1689cdd025bb...
  File:    /content/drive/MyDrive/HF_IQR/v2_planning/HF_IQR_V2_Preregistration.json

Hypotheses locked:    5
Models registered:    6
Questions planned:    200
Categories planned:   12
Scoring formulas:     6
V1 improvements:      12

HF-IQR V2 PRE-REGISTRATION COMPLETE ✅
Status: FILED
No V2 data may be collected before
this timestamp.

Full Force Eternal — Romans 8:28
255.255.255.255:1337


In [6]:
# HF-IQR Phase 0 — W&B Setup
# Real-time experiment tracking
# Free tier sufficient

print("=" * 55)
print("PHASE 1 — W&B SETUP")
print("=" * 55)

# Install W&B
import subprocess
subprocess.run("pip install -q wandb", shell=True)

import wandb
import json
import os

print("\n[1/3] Testing W&B connection...")
print("  You will need your W&B API key from:")
print("  https://wandb.ai/authorize")
print("\n  Run this in your terminal or")
print("  add WANDB_API_KEY to Colab secrets:")
print("\n  wandb.login(key='your_key_here')")

# ── W&B Logger Class ───────────────────────────
print("\n[2/3] Defining W&B logger...")

class WandBLogger:
    """
    Lightweight W&B integration for HF-IQR.
    Tracks cost tokens and scores per round.
    Wraps wandb so the run works without it.
    """

    def __init__(self, project="hf-iqr-v2",
                 run_name=None, enabled=True):
        self.enabled = enabled
        self.project = project
        self.run_name = run_name
        self.run = None

    def start(self, config=None):
        """Initialize W&B run."""
        if not self.enabled:
            return
        try:
            self.run = wandb.init(
                project=self.project,
                name=self.run_name,
                config=config or {},
                resume="allow"
            )
            print(f"  W&B run started ✅")
            print(f"  Dashboard: {self.run.url}")
        except Exception as e:
            print(f"  W&B offline — logging disabled")
            print(f"  Error: {str(e)[:50]}")
            self.enabled = False

    def log_call(self, model, round_num,
                 question_id, tokens,
                 cost, latency_ms):
        """Log one API call."""
        if not self.enabled or not self.run:
            return
        try:
            wandb.log({
                f"{model}/tokens":     tokens,
                f"{model}/cost":       cost,
                f"{model}/latency_ms": latency_ms,
                "round":               round_num,
                "question_id":         question_id
            })
        except:
            pass

    def log_scores(self, model, question_id,
                   esvr=None, dss=None,
                   cvs=None):
        """Log scoring results."""
        if not self.enabled or not self.run:
            return
        try:
            metrics = {}
            if esvr is not None:
                metrics[f"{model}/ESVR"] = esvr
            if dss is not None:
                metrics[f"{model}/DSS"] = dss
            if cvs is not None:
                metrics[f"{model}/CVS"] = cvs
            if metrics:
                wandb.log(metrics)
        except:
            pass

    def log_ri_event(self, question_id,
                     severity, category):
        """Log RI event."""
        if not self.enabled or not self.run:
            return
        try:
            wandb.log({
                "ri_event":    1,
                "ri_severity": severity,
                "ri_category": category,
                "question_id": question_id
            })
        except:
            pass

    def finish(self, summary=None):
        """Close W&B run."""
        if not self.enabled or not self.run:
            return
        try:
            if summary:
                for k, v in summary.items():
                    wandb.run.summary[k] = v
            wandb.finish()
            print(f"  W&B run closed ✅")
        except:
            pass


print("  WandBLogger class defined ✅")

# ── HF Auto Push Function ──────────────────────
print("\n[3/3] Defining HF auto-push...")

def push_to_hf(results_path, dataset_id,
               category=None):
    """
    Auto-push checkpoint data to HF
    after every category run.
    """
    try:
        from datasets import Dataset
        from huggingface_hub import login

        # Load consolidated results
        r1_file = f"{results_path}/round1_all_responses.json"
        if not os.path.exists(r1_file):
            print(f"  No consolidated file yet — skipping HF push")
            return

        with open(r1_file, 'r') as f:
            data = json.load(f)

        responses = data.get("responses", [])
        if not responses:
            print(f"  No responses to push")
            return

        # Convert to HF dataset format
        dataset = Dataset.from_list(responses)
        dataset.push_to_hub(
            dataset_id,
            split=f"round1_{category or 'all'}",
            private=False
        )
        print(f"  Pushed to HF ✅ {dataset_id}")

    except Exception as e:
        print(f"  HF push failed: {str(e)[:50]}")
        print(f"  Manual upload still works")

print("  push_to_hf() defined ✅")

# ── Save config ────────────────────────────────
wandb_config = {
    "project":    "hf-iqr-v2",
    "entity":     "billyrdavis1985",
    "hf_dataset": "Billyrdavis1985/hudson-forge-iqr-v2",
    "enabled":    True
}

config_file = f"{DRIVE_ROOT}/v2_planning/wandb_config.json"
with open(config_file, 'w') as f:
    json.dump(wandb_config, f, indent=2)

print(f"\nW&B config saved ✅")
print(f"\n{'=' * 55}")
print(f"✅ PHASE 1 COMPLETE")
print(f"{'=' * 55}")
print(f"\nPhase 0 + Phase 1 status:")
print(f"  Statistical analysis:  ✅")
print(f"  Human baseline:        ✅")
print(f"  Docker files:          ✅")
print(f"  V2 pre-registration:   ✅")
print(f"  W&B logger:            ✅")
print(f"  HF auto-push:          ✅")
print(f"\n➡️  Next: Phase 2 — V2 dataset construction")
print(f"\nFull Force Eternal — Romans 8:28")
print(f"255.255.255.255:1337")

PHASE 1 — W&B SETUP

[1/3] Testing W&B connection...
  You will need your W&B API key from:
  https://wandb.ai/authorize

  Run this in your terminal or
  add WANDB_API_KEY to Colab secrets:

  wandb.login(key='your_key_here')

[2/3] Defining W&B logger...
  WandBLogger class defined ✅

[3/3] Defining HF auto-push...
  push_to_hf() defined ✅

W&B config saved ✅

✅ PHASE 1 COMPLETE

Phase 0 + Phase 1 status:
  Statistical analysis:  ✅
  Human baseline:        ✅
  Docker files:          ✅
  V2 pre-registration:   ✅
  W&B logger:            ✅
  HF auto-push:          ✅

➡️  Next: Phase 2 — V2 dataset construction

Full Force Eternal — Romans 8:28
255.255.255.255:1337


In [3]:
# HF-IQR Phase 0 — Human Baseline Update
# Session 2 — 12 additional questions

print("=" * 55)
print("HUMAN BASELINE UPDATE — SESSION 2")
print("=" * 55)

import json
import time

# Load existing baseline
baseline_file = (
    f"{RESULTS_PATH}/human_baseline.json")
with open(baseline_file, 'r') as f:
    baseline_data = json.load(f)

existing = baseline_data["responses"]
print(f"Existing responses: {len(existing)}")

# Session 2 responses
session2_responses = [
    {
        "question_id":    "ADV-02",
        "category":       "adversarial",
        "difficulty":     3,
        "answer":         "Ancient Greeks invented many things we use today. Modern humans rely heavily on technology. The trap is that modern humans still accomplish much without technology — the comparison is false.",
        "trap_detected":  True,
        "trap_description": "False comparison — survivorship bias on Greek achievements plus ignoring modern accomplishments",
        "confidence":     100,
        "correct":        True,
        "notes":          "Identified trap cleanly. Could have named survivorship bias specifically."
    },
    {
        "question_id":    "ADV-08",
        "category":       "adversarial",
        "difficulty":     4,
        "answer":         "Strong correlation between chocolate and Nobel prizes. The trap is that correlation does not establish causation — chocolate consumption improves intelligence is not justified.",
        "trap_detected":  True,
        "trap_description": "Correlation to causation leap — wealthy nations consume more chocolate AND produce more Nobel laureates",
        "confidence":     100,
        "correct":        True,
        "notes":          "Correct identification. Could have named confounding variable specifically."
    },
    {
        "question_id":    "PRQ-02",
        "category":       "probabilistic",
        "difficulty":     3,
        "answer":         "Manager found exactly 2 defective parts in 100. The trap is the data supports the conclusion.",
        "trap_detected":  True,
        "trap_description": "Partially detected — identified as trap but concluded data supports conclusion",
        "confidence":     100,
        "correct":        False,
        "notes":          "Growth area. 100 parts insufficient to confirm 2% rate statistically. Need confidence intervals not just point estimate."
    },
    {
        "question_id":    "PRQ-07",
        "category":       "probabilistic",
        "difficulty":     4,
        "answer":         "Pet owners live longer. Insurance considers discount. Not enough data to conclude causation.",
        "trap_detected":  True,
        "trap_description": "Insufficient data — partially detected. Missed reverse causation and confounding variable specifically.",
        "confidence":     100,
        "correct":        "partial",
        "notes":          "Correctly identified causal claim is premature. Missed specific mechanisms: reverse causation and income confounding."
    },
    {
        "question_id":    "QRQ-01",
        "category":       "quantum_reasoning",
        "difficulty":     3,
        "answer":         "Entangled particles are connected. The trap is that particles do not transmit information — nothing moves faster than light.",
        "trap_detected":  True,
        "trap_description": "Correctly identified FTL myth — correlation is instantaneous but no usable information travels",
        "confidence":     100,
        "correct":        True,
        "notes":          "Clean correct answer. Strong quantum reasoning instinct."
    },
    {
        "question_id":    "QRQ-05",
        "category":       "quantum_reasoning",
        "difficulty":     4,
        "answer":         "QC will eventually threaten encryption but timeline is far future. Response should be preparation through treaties not panic.",
        "trap_detected":  True,
        "trap_description": "Correctly identified panic framing as exaggerated — preparation not panic is correct response",
        "confidence":     100,
        "correct":        True,
        "notes":          "Could have named Shor's algorithm and post-quantum cryptography specifically."
    },
    {
        "question_id":    "FRQ-03",
        "category":       "frontier_reasoning",
        "difficulty":     4,
        "answer":         "Connectome mapping does not explain consciousness. The trap is that consciousness is more than neural network information processing.",
        "trap_detected":  True,
        "trap_description": "Correctly identified the hard problem — structural mapping does not explain subjective experience",
        "confidence":     100,
        "correct":        True,
        "notes":          "Strong answer. Could have named the hard problem of consciousness formally."
    },
    {
        "question_id":    "FRQ-07",
        "category":       "frontier_reasoning",
        "difficulty":     4,
        "answer":         "Libet's data shows brain activity before conscious decision. The trap is consciousness is not just along for the ride — memory and consciousness are how decisions are made.",
        "trap_detected":  True,
        "trap_description": "Correctly identified that the experiment does not prove free will is an illusion",
        "confidence":     100,
        "correct":        True,
        "notes":          "Sophisticated answer. Readiness potential may represent preparation not decision. Conscious veto window still exists."
    },
    {
        "question_id":    "CCQ-05",
        "category":       "causal_chain",
        "difficulty":     5,
        "answer":         "Speed cameras installed. Accidents dropped 40%. Researcher is right — not enough data to conclude cameras caused reduction.",
        "trap_detected":  True,
        "trap_description": "Correctly identified insufficient evidence but did not name three specific alternative explanations",
        "confidence":     100,
        "correct":        "partial",
        "notes":          "Growth area. Three specific alternatives: regression to mean secular trend Hawthorne effect. Naming mechanisms required for full credit."
    },
    {
        "question_id":    "LSQ-07",
        "category":       "logical_syllogism",
        "difficulty":     4,
        "answer":         "Neither argument is valid. First says John could win which is too weak. Second says John is a winner which does not follow from being intelligent.",
        "trap_detected":  True,
        "trap_description": "Correctly identified both arguments as invalid and explained the key difference",
        "confidence":     100,
        "correct":        True,
        "notes":          "Could have named affirming the consequent for the second argument."
    },
    {
        "question_id":    "LSQ-09",
        "category":       "logical_syllogism",
        "difficulty":     5,
        "answer":         "No actual argument is stated — cannot conclude soundness without knowing the argument. The argument assumes its own soundness.",
        "trap_detected":  True,
        "trap_description": "Correctly identified circular reasoning — argument assumes what it needs to prove",
        "confidence":     100,
        "correct":        True,
        "notes":          "Deepest answer in set. Named the core problem correctly — begging the question / petitio principii."
    },
    {
        "question_id":    "CCQ-08",
        "category":       "causal_chain",
        "difficulty":     5,
        "answer":         "App users who engage more report higher satisfaction. Three problems: app not described what it does to improve life and does not justify premium price.",
        "trap_detected":  True,
        "trap_description": "Identified causal claim unjustified but named wrong three problems",
        "confidence":     100,
        "correct":        "partial",
        "notes":          "Growth area. Three actual problems: reverse causation selection bias confounding variable. Named surface issues not causal structure."
    }
]

# Combine sessions
all_responses = existing + session2_responses
total         = len(all_responses)
correct       = sum(
    1 for r in all_responses
    if r["correct"] is True)
partial       = sum(
    1 for r in all_responses
    if r["correct"] == "partial")
incorrect     = sum(
    1 for r in all_responses
    if r["correct"] is False)
trap_detected = sum(
    1 for r in all_responses
    if r["trap_detected"])

accuracy = (correct + partial * 0.5) / total
trap_rate = trap_detected / total

print(f"\nCombined baseline results:")
print(f"  Total questions:  {total}")
print(f"  Correct:          {correct}")
print(f"  Partial:          {partial}")
print(f"  Incorrect:        {incorrect}")
print(f"  Trap detected:    "
      f"{trap_detected}/{total}")
print(f"\n  Accuracy:         {accuracy:.1%}")
print(f"  Trap detect rate: {trap_rate:.1%}")

# By category
print(f"\nBy category:")
categories = set(
    r["category"] for r in all_responses)
for cat in sorted(categories):
    cat_results = [
        r for r in all_responses
        if r["category"] == cat]
    cat_correct = sum(
        1 for r in cat_results
        if r["correct"] is True)
    cat_partial = sum(
        1 for r in cat_results
        if r["correct"] == "partial")
    cat_total = len(cat_results)
    cat_score = (
        cat_correct + cat_partial * 0.5)
    print(f"  {cat:<25} "
          f"{cat_score}/{cat_total}")

# Save updated baseline
updated_baseline = {
    "researcher":  "Billy Davis | WARRIOROFGOD40",
    "date_updated": time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    "total":       total,
    "accuracy":    round(accuracy, 4),
    "trap_rate":   round(trap_rate, 4),
    "sessions":    2,
    "responses":   all_responses
}

with open(baseline_file, 'w') as f:
    json.dump(updated_baseline, f, indent=2)

print(f"\nBaseline updated and saved ✅")
print(f"\n{'=' * 55}")
print(f"✅ HUMAN BASELINE COMPLETE")
print(f"{'=' * 55}")
print(f"\n  20 questions answered blind")
print(f"  Accuracy:        {accuracy:.1%}")
print(f"  Trap detection:  {trap_rate:.1%}")
print(f"\n➡️  Next: V2 Council Run Notebook")
print(f"\nFull Force Eternal — Romans 8:28")

HUMAN BASELINE UPDATE — SESSION 2
Existing responses: 8

Combined baseline results:
  Total questions:  20
  Correct:          12
  Partial:          4
  Incorrect:        4
  Trap detected:    19/20

  Accuracy:         70.0%
  Trap detect rate: 95.0%

By category:
  adversarial               2.0/2
  causal_chain              3.0/4
  frontier_reasoning        2.5/4
  logical_syllogism         4.0/4
  probabilistic             0.5/4
  quantum_reasoning         2.0/2

Baseline updated and saved ✅

✅ HUMAN BASELINE COMPLETE

  20 questions answered blind
  Accuracy:        70.0%
  Trap detection:  95.0%

➡️  Next: V2 Council Run Notebook

Full Force Eternal — Romans 8:28
